# Canonical NBA Play-by-Play Dataset

**Goal:** Build a unified play-by-play dataset for backtesting sports betting algorithms.

## The problem

We have two sources of NBA play-by-play data with different schemas:

| Source | API | Coverage | Use case |
|--------|-----|----------|----------|
| **nba_stats** | `stats.nba.com` via `nba_api` SDK (`PlayByPlayV3`) | Historical (full 2024-25 season) | Backtest on past games |
| **nba_cdn** | `cdn.nba.com` REST (no auth) | Live / today's games | Real-time strategy execution |

Both return per-action rows, but the column names and available fields differ. To backtest strategies that will run live, we need a **canonical schema** — the intersection of fields from both sources — so the same code works on historical and live data.

## Approach

1. Explore game-level data (season summary, team records)
2. Pull a sample PBP from each source and compare schemas
3. Identify the common fields and define the canonical schema
4. Build the canonical dataset

In [23]:
import json
import boto3
import pandas as pd

pd.set_option("display.max_columns", None)   # show all columns
pd.set_option("display.width", None)         # auto-detect width
pd.set_option("display.max_colwidth", None)  # full cell content

S3_BUCKET = "prediction-markets-data"
s3 = boto3.client("s3")

## 1. Game-level exploration (nba_stats)

Load the season game data from S3 to understand coverage and pick sample games for PBP comparison.

**S3 bucket:** `s3://prediction-markets-data/`

| Dataset | S3 key pattern | Source |
|---------|---------------|--------|
| Season games | `nba/games/season_{season}.json` | `stats.nba.com` via `nba_api` SDK |
| Play-by-play (historical) | `nba/play_by_play/{game_id}.json` | `stats.nba.com` via `PlayByPlayV3` |
| Play-by-play (live) | `bronze/nba_cdn/live_pbp/YYYY/MM/DD/HH/{uuid}.jsonl.gz` | `cdn.nba.com` via live ingester |
| Scoreboard (live) | `bronze/nba_cdn/scoreboard/YYYY/MM/DD/HH/{uuid}.jsonl.gz` | `cdn.nba.com` via live ingester |
| Odds (live) | `bronze/nba_cdn/odds/YYYY/MM/DD/HH/{uuid}.jsonl.gz` | `cdn.nba.com` via live ingester |
| Box scores (live) | `bronze/nba_cdn/boxscore/YYYY/MM/DD/HH/{uuid}.jsonl.gz` | `cdn.nba.com` via live ingester |

In [24]:
# List what's in the bucket
resp = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix="nba/")
for obj in resp.get("Contents", []):
    print(f"{obj['Key']}  ({obj['Size'] / 1024:.0f} KB)")

nba/games/season_2024-25.json  (1219 KB)
nba/play_by_play/0012400001.json  (270 KB)
nba/play_by_play/0012400002.json  (246 KB)
nba/play_by_play/0012400003.json  (254 KB)
nba/play_by_play/0012400004.json  (268 KB)
nba/play_by_play/0012400005.json  (245 KB)
nba/play_by_play/0012400006.json  (244 KB)
nba/play_by_play/0012400007.json  (247 KB)
nba/play_by_play/0012400008.json  (251 KB)
nba/play_by_play/0012400009.json  (257 KB)
nba/play_by_play/0012400010.json  (251 KB)
nba/play_by_play/0012400011.json  (257 KB)
nba/play_by_play/0012400012.json  (271 KB)
nba/play_by_play/0012400013.json  (236 KB)
nba/play_by_play/0012400014.json  (269 KB)
nba/play_by_play/0012400015.json  (279 KB)
nba/play_by_play/0012400016.json  (279 KB)
nba/play_by_play/0012400017.json  (248 KB)
nba/play_by_play/0012400018.json  (265 KB)
nba/play_by_play/0012400019.json  (251 KB)
nba/play_by_play/0012400020.json  (259 KB)
nba/play_by_play/0012400021.json  (291 KB)
nba/play_by_play/0012400022.json  (251 KB)
nba/play_by_p

In [26]:
# Load a season from S3
SEASON = "2024-25"

obj = s3.get_object(Bucket=S3_BUCKET, Key=f"nba/games/season_{SEASON}.json")
raw = json.loads(obj["Body"].read())
df = pd.DataFrame(raw)
print(f"{len(df)} rows, {df['GAME_ID'].nunique()} games")
df.head()

2802 rows, 1401 games


,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS
0,42024,1610612760,OKC,Oklahoma City Thunder,0042400407,2025-06-22,OKC vs. IND,W,240,103,35,87,0.402,11,40,0.275,22,31,0.710,13,27,40,20,14,8,7,23,12.0
1,42024,1610612754,IND,Indiana Pacers,0042400407,2025-06-22,IND @ OKC,L,240,91,29,70,0.414,11,28,0.393,22,29,0.759,12,33,45,17,6,4,21,24,-12.0
2,42024,1610612760,OKC,Oklahoma City Thunder,0042400406,2025-06-19,OKC @ IND,L,240,91,31,74,0.419,8,30,0.267,21,26,0.808,4,37,41,14,4,4,21,20,-17.0
3,42024,1610612754,IND,Indiana Pacers,0042400406,2025-06-19,IND vs. OKC,W,240,108,38,92,0.413,15,42,0.357,17,25,0.680,11,35,46,23,16,5,10,17,17.0
4,42024,1610612760,OKC,Oklahoma City Thunder,0042400405,2025-06-16,OKC vs. IND,W,239,120,40,94,0.426,14,32,0.438,26,32,0.813,19,26,45,24,15,12,11,24,11.0


In [27]:
# Derive game type from GAME_ID prefix (3rd digit)
# 1 = Preseason, 2 = Regular Season, 3 = All-Star, 4 = Playoffs
game_type_map = {"1": "Preseason", "2": "Regular Season", "3": "All-Star", "4": "Playoffs"}
df["GAME_TYPE"] = df["GAME_ID"].astype(str).str[2].map(game_type_map)
df["GAME_TYPE"].value_counts()

GAME_TYPE
Regular Season    2460
Playoffs           168
Preseason          146
All-Star            14
Name: count, dtype: int64

In [28]:
df.shape

(2802, 29)

In [29]:
df.describe()

,TEAM_ID,MIN,PTS,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS
count,2.802000e+03,2802.000000,2802.000000,2802.000000,2802.000000,2802.000000,2802.000000,2802.000000,2802.000000,2802.000000,2802.000000,2796.000000,2802.000000,2802.000000,2802.000000,2802.000000,2802.000000,2802.000000,2802.000000,2802.000000,2802.000000
mean,1.608314e+09,240.614204,112.940043,41.302641,88.737330,0.466500,13.399714,37.405068,0.357847,16.935046,21.785153,0.777370,11.086010,32.889008,43.975018,26.225910,8.238401,4.885439,13.606353,18.767666,0.001285
std,6.082055e+07,13.355812,13.874430,5.628362,8.359571,0.055139,3.997157,7.268632,0.081508,5.888169,7.120593,0.101171,3.961537,5.639170,6.994675,5.479246,3.130663,2.477764,4.119892,4.428485,16.202305
min,1.502000e+04,39.000000,14.000000,6.000000,19.000000,0.293000,0.000000,8.000000,0.000000,0.000000,0.000000,0.250000,0.000000,3.000000,3.000000,4.000000,0.000000,0.000000,1.000000,0.000000,-59.000000
25%,1.610613e+09,239.000000,104.000000,38.000000,84.000000,0.429000,11.000000,32.000000,0.303000,13.000000,17.000000,0.714000,8.000000,29.000000,39.000000,23.000000,6.000000,3.000000,11.000000,16.000000,-10.000000
50%,1.610613e+09,240.000000,113.000000,41.000000,89.000000,0.466000,13.000000,37.000000,0.357000,17.000000,21.000000,0.783000,11.000000,33.000000,44.000000,26.000000,8.000000,5.000000,13.000000,19.000000,0.000000
75%,1.610613e+09,241.000000,122.000000,45.000000,94.000000,0.505000,16.000000,42.000000,0.412000,21.000000,26.000000,0.846000,14.000000,37.000000,48.000000,30.000000,10.000000,6.000000,16.000000,22.000000,10.000000
max,1.610617e+09,292.000000,162.000000,60.000000,119.000000,0.783000,29.000000,63.000000,0.680000,43.000000,53.000000,1.000000,28.000000,52.000000,73.000000,48.000000,22.000000,18.000000,32.000000,37.000000,59.000000


## 2. Play-by-play schema comparison

The two PBP sources return different schemas. To build a canonical dataset, we need to understand what fields each provides and find the intersection.

### 2a. nba_stats PBP (historical)

Source: `stats.nba.com` via `nba_api.stats.endpoints.playbyplayv3.PlayByPlayV3`
- S3 path: `s3://prediction-markets-data/nba/play_by_play/{game_id}.json`
- Coverage: full 2024-25 season (~690k plays across ~1,401 games)
- Structure: flat list of row dicts (one dict per action)

In [30]:
from nba_api.stats.endpoints import playbyplayv3

# Pick a regular season game
sample_game_id = df.loc[df["GAME_TYPE"] == "Regular Season", "GAME_ID"].iloc[0]
print(f"Game ID: {sample_game_id}")

pbp = playbyplayv3.PlayByPlayV3(game_id=sample_game_id, start_period=1, end_period=10)
pbp_df = pbp.get_data_frames()[0]
print(f"{len(pbp_df)} plays, {pbp_df.shape[1]} columns")
print(f"\nColumns: {pbp_df.columns.tolist()}")
pbp_df.head(10)

Game ID: 0022401195
465 plays, 24 columns

Columns: ['gameId', 'actionNumber', 'clock', 'period', 'teamId', 'teamTricode', 'personId', 'playerName', 'playerNameI', 'xLegacy', 'yLegacy', 'shotDistance', 'shotResult', 'isFieldGoal', 'scoreHome', 'scoreAway', 'pointsTotal', 'location', 'description', 'actionType', 'subType', 'videoAvailable', 'shotValue', 'actionId']


,gameId,actionNumber,clock,period,teamId,teamTricode,personId,playerName,playerNameI,xLegacy,yLegacy,shotDistance,shotResult,isFieldGoal,scoreHome,scoreAway,pointsTotal,location,description,actionType,subType,videoAvailable,shotValue,actionId
0,0022401195,2,PT12M00.00S,1,0,,0,,,0,0,0,,0,0,0,0,,Start of 1st Period (3:41 PM EST),period,start,0,0,1
1,0022401195,4,PT12M00.00S,1,1610612750,MIN,203497,Gobert,R. Gobert,0,0,0,,0,,,0,h,Jump Ball Gobert vs. Filipowski: Tip to Juzang,Jump Ball,,1,0,2
2,0022401195,7,PT11M38.00S,1,1610612762,UTA,1642271,Filipowski,K. Filipowski,16,247,25,Made,1,0,3,3,v,Filipowski 25' 3PT Step Back Jump Shot (3 PTS) (Sensabaugh 1 AST),Made Shot,Step Back Jump shot,1,3,3
3,0022401195,9,PT11M17.00S,1,1610612750,MIN,1630162,Edwards,A. Edwards,104,228,25,Missed,1,,,0,h,MISS Edwards 25' 3PT Pullup Jump Shot,Missed Shot,Pullup Jump shot,1,3,4
4,0022401195,10,PT11M14.00S,1,1610612762,UTA,1641729,Sensabaugh,B. Sensabaugh,0,0,0,,0,,,0,v,Sensabaugh REBOUND (Off:0 Def:1),Rebound,Unknown,1,0,5
5,0022401195,11,PT10M54.00S,1,1610612762,UTA,1641718,George,K. George,-188,153,24,Missed,1,,,0,v,MISS George 24' 3PT Step Back Jump Shot,Missed Shot,Step Back Jump shot,1,3,6
6,0022401195,12,PT10M50.00S,1,1610612750,MIN,203944,Randle,J. Randle,0,0,0,,0,,,0,h,Randle REBOUND (Off:0 Def:1),Rebound,Unknown,1,0,7
7,0022401195,13,PT10M45.00S,1,1610612750,MIN,1630183,McDaniels,J. McDaniels,-1,4,0,Made,1,2,3,5,h,McDaniels Driving Layup (2 PTS) (Conley 1 AST),Made Shot,Driving Layup Shot,1,2,8
8,0022401195,15,PT10M36.00S,1,1610612762,UTA,1630548,Juzang,J. Juzang,-1,9,1,Made,1,2,5,7,v,Juzang 1' Cutting Layup Shot (2 PTS) (Sensabaugh 2 AST),Made Shot,Cutting Layup Shot,1,2,9
9,0022401195,17,PT10M24.00S,1,1610612750,MIN,201144,Conley,M. Conley,-89,247,26,Missed,1,,,0,h,MISS Conley 26' 3PT Pullup Jump Shot,Missed Shot,Pullup Jump shot,1,3,10


### 2b. nba_cdn PBP (live)

Source: `cdn.nba.com/static/json/liveData/playbyplay/playbyplay_{game_id}.json`
- S3 path: `s3://prediction-markets-data/bronze/nba_cdn/live_pbp/YYYY/MM/DD/HH/{uuid}.jsonl.gz`
- Format: gzip-compressed JSONL, one action per line, raw action nested under `frame` key
- Coverage: games captured while the live ingester was running
- Wrapper fields per line: `source`, `channel`, `game_id`, `action_number`, `t_request`, `t_receipt`, `poll_seq`, `frame`

In [43]:
import gzip

# List bronze live_pbp files
resp = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix="bronze/nba_cdn/live_pbp/")
bronze_keys = [obj["Key"] for obj in resp.get("Contents", [])]
print(f"{len(bronze_keys)} bronze live_pbp files in S3")

if bronze_keys:
    # Read multiple files to get a fuller picture — each file is one flush (~1 min of actions)
    all_actions = []
    for key in bronze_keys[:5]:
        obj = s3.get_object(Bucket=S3_BUCKET, Key=key)
        raw_bytes = gzip.decompress(obj["Body"].read())
        for line in raw_bytes.decode().strip().split("\n"):
            all_actions.append(json.loads(line)["frame"])
    cdn_pbp_df = pd.DataFrame(all_actions)
    print(f"Read {len(bronze_keys[:20])} files, {len(cdn_pbp_df)} total actions, {cdn_pbp_df.shape[1]} columns")
    print(f"\nColumns: {cdn_pbp_df.columns.tolist()}")
    cdn_pbp_df.head(10)
else:
    print("No bronze live_pbp data in S3 yet — run: python -m scripts.live.nba_cdn")

401 bronze live_pbp files in S3
Read 20 files, 21 total actions, 44 columns

Columns: ['actionNumber', 'clock', 'timeActual', 'period', 'periodType', 'teamId', 'teamTricode', 'actionType', 'qualifiers', 'personId', 'x', 'y', 'area', 'areaDetail', 'side', 'shotDistance', 'possession', 'scoreHome', 'scoreAway', 'edited', 'orderNumber', 'isTargetScoreLastPeriod', 'subType', 'xLegacy', 'yLegacy', 'isFieldGoal', 'shotResult', 'pointsTotal', 'description', 'playerName', 'playerNameI', 'personIdsFilter', 'descriptor', 'shotActionNumber', 'reboundTotal', 'reboundDefensiveTotal', 'reboundOffensiveTotal', 'assistPlayerNameInitial', 'assistPersonId', 'assistTotal', 'foulPersonalTotal', 'foulTechnicalTotal', 'foulDrawnPlayerName', 'foulDrawnPersonId']


In [44]:
cdn_pbp_df

,actionNumber,clock,timeActual,period,periodType,teamId,teamTricode,actionType,qualifiers,personId,x,y,area,areaDetail,side,shotDistance,possession,scoreHome,scoreAway,edited,orderNumber,isTargetScoreLastPeriod,subType,xLegacy,yLegacy,isFieldGoal,shotResult,pointsTotal,description,playerName,playerNameI,personIdsFilter,descriptor,shotActionNumber,reboundTotal,reboundDefensiveTotal,reboundOffensiveTotal,assistPlayerNameInitial,assistPersonId,assistTotal,foulPersonalTotal,foulTechnicalTotal,foulDrawnPlayerName,foulDrawnPersonId
0,188,PT08M21.00S,2026-04-18T22:54:14.2Z,2,REGULAR,1610612737,ATL,2pt,[pointsinthepaint],203468,5.995401,48.284314,Restricted Area,0-8 Center,left,0.94,1610612737,42,34,2026-04-18T22:54:15Z,1850000,False,shot,9.0,4.0,1,Made,10.0,C. McCollum shot (10 PTS),McCollum,C. McCollum,[203468],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,189,PT08M00.00S,2026-04-18T22:54:35.5Z,2,REGULAR,1610612752,NYK,2pt,[pointsinthepaint],1629013,92.723390,52.696078,Restricted Area,0-8 Center,right,2.09,1610612752,42,34,2026-04-18T22:54:39Z,1860000,False,Layup,13.0,16.0,1,Missed,NaN,MISS L. Shamet driving Layup,Shamet,L. Shamet,[1629013],driving,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,190,PT07M59.00S,2026-04-18T22:54:36.5Z,2,REGULAR,1610612737,ATL,rebound,[team],0,NaN,NaN,Restricted Area,0-8 Center,NaN,NaN,1610612737,42,34,2026-04-18T22:54:39Z,1870000,False,defensive,NaN,NaN,0,NaN,NaN,TEAM defensive REBOUND,NaN,NaN,[],NaN,189.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,94,PT03M27.00S,2026-04-18T22:37:56.8Z,1,REGULAR,1610612752,NYK,2pt,[pointsinthepaint],1628973,82.210907,43.137255,In The Paint (Non-RA),8-16 Center,right,11.97,1610612752,22,21,2026-04-18T22:38:00Z,910000,False,Jump Shot,-34.0,115.0,1,Missed,NaN,MISS J. Brunson 11' turnaround Shot,Brunson,J. Brunson,[1628973],turnaround,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,95,PT03M24.00S,2026-04-18T22:37:59.8Z,1,REGULAR,1610612737,ATL,rebound,[],1630228,NaN,NaN,In The Paint (Non-RA),8-16 Center,NaN,NaN,1610612737,22,21,2026-04-18T22:38:00Z,920000,False,defensive,NaN,NaN,0,NaN,NaN,J. Kuminga REBOUND (Off:0 Def:1),Kuminga,J. Kuminga,[1630228],NaN,94.0,1.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,96,PT03M13.00S,2026-04-18T22:38:10.4Z,1,REGULAR,1610612737,ATL,2pt,[pointsinthepaint],1630168,6.389619,47.794118,Restricted Area,0-8 Center,left,1.34,1610612737,22,23,2026-04-18T22:38:11Z,930000,False,shot,11.0,8.0,1,Made,4.0,O. Okongwu shot (4 PTS) (N. Alexander-Walker 3 AST),Okongwu,O. Okongwu,"[1630168, 1629638]",NaN,NaN,NaN,NaN,NaN,N. Alexander-Walker,1629638.0,3.0,NaN,NaN,NaN,NaN
6,98,PT02M51.00S,2026-04-18T22:38:32.3Z,1,REGULAR,1610612752,NYK,3pt,[],1630540,72.618265,15.441176,Above the Break 3,24+ Left Center,right,26.80,1610612752,22,23,2026-04-18T22:38:39Z,950000,False,Jump Shot,-173.0,205.0,1,Missed,NaN,MISS M. McBride 26' pullup 3PT,McBride,M. McBride,[1630540],pullup,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,99,PT02M49.00S,2026-04-18T22:38:34.3Z,1,REGULAR,1610612737,ATL,rebound,[],1630552,NaN,NaN,Above the Break 3,24+ Left Center,NaN,NaN,1610612737,22,23,2026-04-18T22:38:39Z,960000,False,defensive,NaN,NaN,0,NaN,NaN,J. Johnson REBOUND (Off:0 Def:2),Johnson,J. Johnson,[1630552],NaN,98.0,2.0,2.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,100,PT02M37.00S,2026-04-18T22:38:46.1Z,1,REGULAR,1610612737,ATL,2pt,[pointsinthepaint],1630228,8.229304,64.950980,In The Paint (Non-RA),0-8 Center,left,7.88,1610612737,22,23,2026-04-18T22:38:47Z,970000,False,shot,-75.0,25.0,1,Missed,NaN,MISS J. Kuminga 7' shot,Kuminga,J. Kuminga,[1630228],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,101,PT02M36.00S,2026-04-18T22:38:48.5Z,1,REGULAR,1610612752,NYK,rebound,[],1628384,NaN,NaN,In The Paint (Non-RA),0-8 Center,NaN,NaN,1610612752,22,23,2026-04-18T22:38:48Z,980000,False,defensive,NaN,NaN,0,NaN,NaN,O. Anunoby REBOUND (Off:0 Def:2),Anunoby,O. Anunoby,[1628384],NaN,100.0,2.0,2.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [34]:
# Explore CDN action types and structure
print("Action types:")
print(cdn_pbp_df["actionType"].value_counts())
print(f"\nDtypes:\n{cdn_pbp_df.dtypes}")

Action types:
actionType
2pt        2
rebound    1
Name: count, dtype: int64

Dtypes:
actionNumber                 int64
clock                          str
timeActual                     str
period                       int64
periodType                     str
teamId                       int64
teamTricode                    str
actionType                     str
qualifiers                  object
personId                     int64
x                          float64
y                          float64
area                           str
areaDetail                     str
side                           str
shotDistance               float64
possession                   int64
scoreHome                      str
scoreAway                      str
edited                         str
orderNumber                  int64
isTargetScoreLastPeriod       bool
subType                        str
xLegacy                    float64
yLegacy                    float64
isFieldGoal                  int64
shot

### 2c. Schema comparison

Compare the column sets from both sources to find the intersection.

In [35]:
stats_cols = set(pbp_df.columns)
cdn_cols = set(cdn_pbp_df.columns) if bronze_keys else set()

print("=== nba_stats only ===")
print(sorted(stats_cols - cdn_cols))

print("\n=== nba_cdn only ===")
print(sorted(cdn_cols - stats_cols))

print("\n=== Common (intersection) ===")
common = sorted(stats_cols & cdn_cols)
print(common)
print(f"\n{len(common)} common fields out of {len(stats_cols)} stats / {len(cdn_cols)} cdn")

=== nba_stats only ===
['actionId', 'gameId', 'location', 'shotValue', 'videoAvailable']

=== nba_cdn only ===
['area', 'areaDetail', 'descriptor', 'edited', 'isTargetScoreLastPeriod', 'orderNumber', 'periodType', 'personIdsFilter', 'possession', 'qualifiers', 'shotActionNumber', 'side', 'timeActual', 'x', 'y']

=== Common (intersection) ===
['actionNumber', 'actionType', 'clock', 'description', 'isFieldGoal', 'period', 'personId', 'playerName', 'playerNameI', 'pointsTotal', 'scoreAway', 'scoreHome', 'shotDistance', 'shotResult', 'subType', 'teamId', 'teamTricode', 'xLegacy', 'yLegacy']

19 common fields out of 24 stats / 34 cdn


## 3. Kalshi market data — can we join on game state?

To backtest betting strategies, we need to correlate NBA game state (score, period, clock) with Kalshi market prices. The question is: does Kalshi data contain game state fields we can join on, or do we need to join on wall-clock time?

**Kalshi data in S3:**

| Dataset | S3 key pattern | Key fields |
|---------|---------------|------------|
| Historical markets | `kalshi/historical_markets/{series}.json` | `ticker`, `event_ticker`, `title`, `open_time`, `close_time`, `result` |
| Historical trades | `kalshi/historical_trades/{ticker}.json` | `ticker`, `price`, `count`, `taker_side`, `created_time` |
| Historical candlesticks | `kalshi/historical_candlesticks/{interval}m/{ticker}.json` | OHLC + `volume` + `timestamp` |
| Live orderbook snapshots | `bronze/kalshi_ws/orderbook_snapshot/YYYY/MM/DD/HH/{uuid}.jsonl.gz` | `market_ticker`, `bids`, `asks` |
| Live orderbook deltas | `bronze/kalshi_ws/orderbook_delta/YYYY/MM/DD/HH/{uuid}.jsonl.gz` | `market_ticker`, price/size deltas |
| Live trades | `bronze/kalshi_ws/trade/YYYY/MM/DD/HH/{uuid}.jsonl.gz` | `market_ticker`, `price`, `count`, `taker_side`, `created_time` |

In [47]:
# Load a sample historical markets file (e.g. KXNBAGAME — win/loss)
obj = s3.get_object(Bucket=S3_BUCKET, Key="kalshi/historical_markets/KXNBAGAME.json")
markets_raw = json.loads(obj["Body"].read())
markets_df = pd.DataFrame(markets_raw)
print(f"{len(markets_df)} markets")
print(f"\nColumns: {markets_df.columns.tolist()}")
markets_df.head(2)

1902 markets

Columns: ['can_close_early', 'close_time', 'created_time', 'custom_strike', 'early_close_condition', 'event_ticker', 'expected_expiration_time', 'expiration_time', 'expiration_value', 'fractional_trading_enabled', 'last_price_dollars', 'latest_expiration_time', 'liquidity_dollars', 'market_type', 'no_ask_dollars', 'no_bid_dollars', 'no_sub_title', 'notional_value_dollars', 'open_interest_fp', 'open_time', 'previous_price_dollars', 'previous_yes_ask_dollars', 'previous_yes_bid_dollars', 'price_level_structure', 'price_ranges', 'response_price_units', 'result', 'rules_primary', 'rules_secondary', 'settlement_timer_seconds', 'settlement_ts', 'settlement_value_dollars', 'status', 'strike_type', 'tick_size', 'ticker', 'title', 'updated_time', 'volume_24h_fp', 'volume_fp', 'yes_ask_dollars', 'yes_ask_size_fp', 'yes_bid_dollars', 'yes_bid_size_fp', 'yes_sub_title']


,can_close_early,close_time,created_time,custom_strike,early_close_condition,event_ticker,expected_expiration_time,expiration_time,expiration_value,fractional_trading_enabled,last_price_dollars,latest_expiration_time,liquidity_dollars,market_type,no_ask_dollars,no_bid_dollars,no_sub_title,notional_value_dollars,open_interest_fp,open_time,previous_price_dollars,previous_yes_ask_dollars,previous_yes_bid_dollars,price_level_structure,price_ranges,response_price_units,result,rules_primary,rules_secondary,settlement_timer_seconds,settlement_ts,settlement_value_dollars,status,strike_type,tick_size,ticker,title,updated_time,volume_24h_fp,volume_fp,yes_ask_dollars,yes_ask_size_fp,yes_bid_dollars,yes_bid_size_fp,yes_sub_title
0,True,2026-02-13T05:37:28Z,2026-02-10T17:02:09.225173Z,{'basketball_team': '2ef4d31c-0b46-4f43-a403-f44d62489034'},This market will close and expire after a winner is declared.,KXNBAGAME-26FEB12DALLAL,2026-02-13T06:00:00Z,2026-02-27T03:00:00Z,Los Angeles L,False,0.9900,2026-02-27T03:00:00Z,0.0000,binary,1.0000,0.0000,Los Angeles L,1.0000,0.00,2026-02-10T22:06:00Z,0.9900,1.0000,0.0000,linear_cent,"[{'end': '1.0000', 'start': '0.0000', 'step': '0.0100'}]",usd_cent,yes,"If Los Angeles L wins the Dallas at Los Angeles L professional basketball game originally scheduled for Feb 12, 2026, then the market resolves to Yes.",,30,2026-02-13T05:39:01.690752Z,1.0000,finalized,structured,1,KXNBAGAME-26FEB12DALLAL-LAL,Dallas at Los Angeles L Winner?,2026-02-19T08:48:16.628714Z,0.00,4954411.00,1.0000,0.00,0.0000,0.00,Los Angeles L
1,True,2026-02-13T05:37:28Z,2026-02-10T17:02:09.225173Z,{'basketball_team': '36547ae6-f0fa-4d06-a6e5-5bfa64a217ed'},This market will close and expire after a winner is declared.,KXNBAGAME-26FEB12DALLAL,2026-02-13T06:00:00Z,2026-02-27T03:00:00Z,Los Angeles L,False,0.0100,2026-02-27T03:00:00Z,0.0000,binary,1.0000,0.0000,Dallas,1.0000,0.00,2026-02-10T22:06:00Z,0.0100,1.0000,0.0000,linear_cent,"[{'end': '1.0000', 'start': '0.0000', 'step': '0.0100'}]",usd_cent,no,"If Dallas wins the Dallas at Los Angeles L professional basketball game originally scheduled for Feb 12, 2026, then the market resolves to Yes.",,30,2026-02-13T05:39:01.690752Z,0.0000,finalized,structured,1,KXNBAGAME-26FEB12DALLAL-DAL,Dallas at Los Angeles L Winner?,2026-02-19T08:48:16.628714Z,0.00,3744380.00,1.0000,0.00,0.0000,0.00,Dallas


In [48]:
# Load trades for one ticker to see the schema
sample_ticker = markets_df["ticker"].iloc[0]
print(f"Sample ticker: {sample_ticker}")

obj = s3.get_object(Bucket=S3_BUCKET, Key=f"kalshi/historical_trades/{sample_ticker}.json")
trades_raw = json.loads(obj["Body"].read())
trades_df = pd.DataFrame(trades_raw)
print(f"{len(trades_df)} trades")
print(f"\nColumns: {trades_df.columns.tolist()}")
trades_df.head(10)

Sample ticker: KXNBAGAME-26FEB12DALLAL-LAL
22533 trades

Columns: ['count_fp', 'created_time', 'no_price_dollars', 'taker_side', 'ticker', 'trade_id', 'yes_price_dollars']


,count_fp,created_time,no_price_dollars,taker_side,ticker,trade_id,yes_price_dollars
0,15.00,2026-02-13T05:30:39.335643Z,0.0100,no,KXNBAGAME-26FEB12DALLAL-LAL,3f8fa67d-6fbf-40e0-5f40-6fe027f593a3,0.9900
1,52.00,2026-02-13T05:30:37.195315Z,0.0100,no,KXNBAGAME-26FEB12DALLAL-LAL,973b978b-87bf-4a28-fd3c-1acc939fdd77,0.9900
2,20.00,2026-02-13T05:30:34.13469Z,0.0100,no,KXNBAGAME-26FEB12DALLAL-LAL,39ffa7cd-b71f-5b80-1dfa-529b3df1485b,0.9900
3,6.00,2026-02-13T05:30:33.604435Z,0.0100,no,KXNBAGAME-26FEB12DALLAL-LAL,c6333019-83a7-561c-eba4-748526b6b773,0.9900
4,19.00,2026-02-13T05:30:32.710968Z,0.0100,no,KXNBAGAME-26FEB12DALLAL-LAL,7074d157-0383-4bc8-2f9c-38e05e56698f,0.9900
5,34.00,2026-02-13T05:30:29.459328Z,0.0100,no,KXNBAGAME-26FEB12DALLAL-LAL,c0ab09ec-4eeb-4b74-d514-fae2aea936d3,0.9900
6,5.00,2026-02-13T05:30:26.833671Z,0.0100,no,KXNBAGAME-26FEB12DALLAL-LAL,ac00f367-62b7-6bc8-6530-2e8d3a15ce93,0.9900
7,2710.00,2026-02-13T05:30:26.201774Z,0.0100,no,KXNBAGAME-26FEB12DALLAL-LAL,24a152f8-8a2f-5dc4-df15-35c374055bc7,0.9900
8,252.00,2026-02-13T05:30:21.131704Z,0.0100,no,KXNBAGAME-26FEB12DALLAL-LAL,283c7485-cd8b-6c60-69a4-ffb1d34d9cbf,0.9900
9,10.00,2026-02-13T05:30:19.673164Z,0.0100,no,KXNBAGAME-26FEB12DALLAL-LAL,860609da-556f-45bc-6724-2a89b0bb818f,0.9900


In [49]:
# Load live Kalshi WS trades from bronze
resp = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix="bronze/kalshi_ws/trade/")
kalshi_bronze_keys = [obj["Key"] for obj in resp.get("Contents", [])]
print(f"{len(kalshi_bronze_keys)} bronze kalshi_ws/trade files in S3")

if kalshi_bronze_keys:
    all_ws_trades = []
    for key in kalshi_bronze_keys[:20]:
        obj = s3.get_object(Bucket=S3_BUCKET, Key=key)
        raw_bytes = gzip.decompress(obj["Body"].read())
        for line in raw_bytes.decode().strip().split("\n"):
            record = json.loads(line)
            frame = record["frame"]
            # WS trade frames have msg nested inside
            msg = frame.get("msg", {})
            msg["t_receipt"] = record.get("t_receipt")
            all_ws_trades.append(msg)
    ws_trades_df = pd.DataFrame(all_ws_trades)
    print(f"Read {len(kalshi_bronze_keys[:20])} files, {len(ws_trades_df)} trades")
    print(f"\nColumns: {ws_trades_df.columns.tolist()}")
    ws_trades_df.head(10)
else:
    print("No bronze kalshi_ws/trade data yet — run: python -m scripts.live.kalshi_ws")

1000 bronze kalshi_ws/trade files in S3
Read 20 files, 20723 trades

Columns: ['trade_id', 'market_ticker', 'yes_price_dollars', 'no_price_dollars', 'count_fp', 'taker_side', 'ts', 't_receipt']


### 3a. Joining Kalshi to NBA — observations

Kalshi data has **no game state fields** (`period`, `clock`, `score`). It is purely market data. The only way to correlate market prices with game state is:

1. **Map ticker to game**: parse `event_ticker` (e.g. `KXNBAGAME-25APR18-LALBOS-YES`) to extract date + teams, then match to an NBA `game_id`
2. **Join on wall-clock time**: Kalshi trades have `created_time`, NBA CDN has `timeActual` — align by timestamp to answer "what was the market price when this play happened?"

This means `timeActual` is essential for the join. Since nba_stats (historical) doesn't have it, our options are:
- **Live data only**: use CDN `timeActual` + Kalshi `created_time` (available going forward)
- **Historical approximation**: for nba_stats, the `description` field sometimes contains the game start time (e.g. "Start of 1st Period (3:41 PM EST)") — we could parse this and reconstruct approximate wall-clock times from `period` + `clock`, though stoppages make this inexact